In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema
from sklearn.neighbors import NearestNeighbors
import os
import csv

# ==========================================
# --- Helper Functions for Parameter Estimation ---
# ==========================================

def average_mutual_information(x, max_lag=100, bins=64):
    """
    Calculate the average mutual information for a time series.
    Finds the first local minimum to suggest an optimal delay `tau`.
    """
    x = np.asarray(x)
    ami_values = np.zeros(max_lag)
    for lag in range(1, max_lag + 1):
        if lag >= len(x):
            ami_values[lag - 1] = np.nan
            continue
        x1 = x[:-lag]
        x2 = x[lag:]

        hist_2d, _, _ = np.histogram2d(x1, x2, bins=bins)
        pxy = hist_2d / np.sum(hist_2d)
        px = np.sum(pxy, axis=1)
        py = np.sum(pxy, axis=0)
        px_py = np.outer(px, py)
        nzs = pxy > 0
        mi = np.sum(pxy[nzs] * np.log(pxy[nzs] / px_py[nzs]))
        ami_values[lag - 1] = mi

    return ami_values


def find_optimal_delay(time_series):
    ami = average_mutual_information(time_series)
    local_min_indices = argrelextrema(ami, np.less)[0]
    if len(local_min_indices) > 0:
        return local_min_indices[0] + 1
    try:
        return np.where(ami < ami[0] / np.e)[0][0] + 1
    except IndexError:
        return 1


def delay_embedding_aligned(x, m, tau):
    x = np.asarray(x)
    n_vectors = len(x) - (m + 1) * tau + 1
    if n_vectors <= 0:
        return np.empty((0, m)), np.empty((0, m + 1))
    Y_m = np.array([x[i:i + m * tau:tau] for i in range(n_vectors)])
    Y_mp1 = np.array([x[i:i + (m + 1) * tau:tau] for i in range(n_vectors)])
    return Y_m, Y_mp1


def false_nearest_neighbors_fraction(x, tau, max_dim=10, rtol=10.0, atol=2.0):
    fnn_fractions = []
    x_std = np.std(x)
    for m in range(1, max_dim + 1):
        if len(x) <= (m + 1) * tau:
            fnn_fractions.append(np.nan)
            continue

        Y_m, Y_mp1 = delay_embedding_aligned(x, m, tau)
        if Y_m.shape[0] == 0:
            fnn_fractions.append(np.nan)
            continue

        nbrs = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(Y_m)
        distances, indices = nbrs.kneighbors(Y_m)
        dist_m = distances[:, 1]
        neighbors_idx = indices[:, 1]
        dist_mp1 = np.linalg.norm(Y_mp1 - Y_mp1[neighbors_idx], axis=1)

        with np.errstate(divide='ignore', invalid='ignore'):
            cond1 = dist_mp1 / dist_m > rtol
            cond2 = dist_mp1 / x_std > atol
        false_neighbors = np.logical_or(cond1, cond2)
        valid_indices = dist_m > 0
        fnn_fraction = np.sum(false_neighbors[valid_indices]) / np.sum(valid_indices) if np.sum(valid_indices) > 0 else 1.0
        fnn_fractions.append(fnn_fraction)
    return np.array(fnn_fractions)


def find_optimal_dimension(time_series, tau):
    fnn_fracs = false_nearest_neighbors_fraction(time_series, tau=tau)
    try:
        return np.where(fnn_fracs < 0.1)[0][0] + 1
    except IndexError:
        return np.argmin(fnn_fracs) + 1


# ==========================================
# --- Main Script ---
# ==========================================

if __name__ == '__main__':
    stocks = ['AAPL', 'ADBE', 'AMAT', 'AMZN', 'CSCO', 'GOOGL', 'INTC', 'INTU',
              'KLAC', 'LRCX', 'META', 'MSFT', 'MU', 'NFLX', 'NVDA', 'QCOM', 'TSLA', 'TXN']
    data_dir = 'clean_data'
    plot_dir = 'geminiplots'
    results = []

    os.makedirs(plot_dir, exist_ok=True)

    if not os.path.exists(data_dir):
        print(f"Error: Directory '{data_dir}' not found.")
        exit()

    for stock in stocks:
        file_path = os.path.join(data_dir, f"{stock}_clean.csv")
        if not os.path.exists(file_path):
            print(f"\n--- {stock}: File not found. Skipping. ---")
            continue

        print(f"\n--- Analyzing {stock} ---")

        try:
            df = pd.read_csv(file_path)
            time_series = df['Log_Return_Clean'].values

            # --- Compute τ (delay) ---
            tau_opt = find_optimal_delay(time_series)

            # --- Compute m (embedding dimension) ---
            m_opt = find_optimal_dimension(time_series, tau=tau_opt)

            print(f"Optimal Delay (τ): {tau_opt}, Optimal Embedding Dimension (m): {m_opt}")

            # --- Save results ---
            results.append({'Stock': stock, 'Optimal_Delay_Tau': tau_opt, 'Optimal_Embedding_m': m_opt})

            # --- Plot AMI ---
            ami = average_mutual_information(time_series)
            plt.figure(figsize=(6, 4))
            plt.plot(range(1, len(ami) + 1), ami, '-o')
            plt.axvline(tau_opt, color='r', linestyle='--', label=f'Optimal τ={tau_opt}')
            plt.title(f'Average Mutual Information - {stock}')
            plt.xlabel('Lag')
            plt.ylabel('AMI')
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(plot_dir, f'{stock}_AMI.png'))
            plt.close()

            # --- Plot FNN ---
            fnn_fracs = false_nearest_neighbors_fraction(time_series, tau=tau_opt)
            plt.figure(figsize=(6, 4))
            plt.plot(range(1, len(fnn_fracs) + 1), fnn_fracs, '-o')
            plt.axvline(m_opt, color='r', linestyle='--', label=f'Optimal m={m_opt}')
            plt.title(f'False Nearest Neighbors - {stock}')
            plt.xlabel('Embedding Dimension (m)')
            plt.ylabel('FNN Fraction')
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(plot_dir, f'{stock}_FNN.png'))
            plt.close()

        except Exception as e:
            print(f"Error processing {stock}: {e}")

    # --- Save results as CSV ---
    output_file = 'embedding_params.csv'
    with open(output_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['Stock', 'Optimal_Delay_Tau', 'Optimal_Embedding_m'])
        writer.writeheader()
        writer.writerows(results)
    print(f"\nSaved results to {output_file}")

    # --- Print LaTeX Table ---
    df_results = pd.DataFrame(results)
    print("\nLaTeX Table:\n")
    print(df_results.to_latex(index=False, float_format="%.2f"))



--- Analyzing AAPL ---
Optimal Delay (τ): 6, Optimal Embedding Dimension (m): 5

--- Analyzing ADBE ---
Optimal Delay (τ): 7, Optimal Embedding Dimension (m): 5

--- Analyzing AMAT ---
Optimal Delay (τ): 6, Optimal Embedding Dimension (m): 6

--- Analyzing AMZN ---
Optimal Delay (τ): 5, Optimal Embedding Dimension (m): 7

--- Analyzing CSCO ---
Optimal Delay (τ): 5, Optimal Embedding Dimension (m): 4

--- Analyzing GOOGL ---
Optimal Delay (τ): 8, Optimal Embedding Dimension (m): 7

--- Analyzing INTC ---
Optimal Delay (τ): 6, Optimal Embedding Dimension (m): 5

--- Analyzing INTU ---
Optimal Delay (τ): 5, Optimal Embedding Dimension (m): 5

--- Analyzing KLAC ---
Optimal Delay (τ): 7, Optimal Embedding Dimension (m): 7

--- Analyzing LRCX ---
Optimal Delay (τ): 6, Optimal Embedding Dimension (m): 5

--- Analyzing META ---
Optimal Delay (τ): 5, Optimal Embedding Dimension (m): 5

--- Analyzing MSFT ---
Optimal Delay (τ): 6, Optimal Embedding Dimension (m): 5

--- Analyzing MU ---
Optim